<a href="https://colab.research.google.com/github/Hudiii/20solver_web/blob/main/Optimasi%20DP%20dan%20test%20Greedy%2070%3A30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Import Library

In [ ]:
import time
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Ambil Dataset

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Dataset_Wisata_Solo.csv")

# Konversi tipe data agar cost = int, rating = float
df["Biaya"] = (df["Biaya"]/1000).astype(int)
df["Rating"] = df["Rating"].astype(float)

# Ubah ke format list of tuples: (Nama, Biaya, Rating)
places = [(row["Nama"], int(row["Biaya"]), float(row["Rating"]))
          for _, row in df.iterrows()]

# Preprocessing

In [ ]:
free_places = [p for p in places if p[1] == 0]
paid_places = [p for p in places if p[1] > 0]

##Algoritma Dynamic Programming

In [ ]:
def dp_best_tour(places, max_cost):
    start = time.time()
    n = len(places)

    # dp[budget] = (rating, cost, list_of_indices)
    dp = [(0, 0, []) for _ in range(max_cost + 1)]

    for idx, (name, cost, rating) in enumerate(places):
        for budget in range(max_cost, cost - 1, -1):  # iterate backwards to avoid reuse in same iteration
            prev_rating, prev_cost, prev_indices = dp[budget - cost]

            new_rating = prev_rating + rating
            new_cost = prev_cost + cost

            if (new_rating > dp[budget][0]) or \
               (new_rating == dp[budget][0] and new_cost < dp[budget][1]):
                dp[budget] = (new_rating, new_cost, prev_indices + [idx])

    best = max(dp, key=lambda x: (x[0], -x[1]))
    exec_time = time.time() - start

    # Reconstruct names and detail list
    places_selected = [places[i][0] for i in best[2]]
    details = [(places[i][0], places[i][1], places[i][2]) for i in best[2]]

    return best[0], best[1], places_selected, details, exec_time


##Algoritma Greedy

In [ ]:
def greedy_best_tour(places, max_cost):
    start = time.time()

    def score(p):
        name, cost, rating = p
        return float('inf') if cost == 0 else (0.7 * rating + 0.3 * (rating / cost))

    data = sorted(places, key=score, reverse=True)

    total_rating = 0
    total_cost = 0
    selected = []
    details = []

    for name, cost, rating in data:
        if total_cost + cost <= max_cost:
            total_cost += cost
            total_rating += rating
            selected.append(name)
            details.append((name, cost, rating))

            # Adaptive improvement: resort sisa data berdasarkan cost-efficiency
            data = sorted(data, key=score, reverse=True)

    exec_time = time.time() - start
    return total_rating, total_cost, selected, details, exec_time


##Input, Eksekusi, dan Hasil

In [ ]:
max_cost = int(input("Masukkan budget maksimal (dalam ribuan): "))

dp_rating, dp_cost, dp_places, dp_details, dp_time = dp_best_tour(paid_places, max_cost)
gr_rating, gr_cost, gr_places, gr_details, gr_time = greedy_best_tour(paid_places, max_cost)

print("\n HASIL PERBANDINGAN DP vs GREEDY ")
print(f"\nDynamic Programming:")
print(f"Total Rating: {dp_rating:.2f}, Total Cost: Rp {dp_cost}000, Time: {dp_time:.5f}s")
print("Destinasi:", ", ".join(dp_places))

print(f"\nGreedy:")
print(f"Total Rating: {gr_rating:.2f}, Total Cost: Rp {gr_cost}000, Time: {gr_time:.5f}s")
print("Destinasi:", ", ".join(gr_places))

print("\n DESTINASI GRATIS (bonus) ")
for name, cost, rating in free_places:
    print(f"{name} (Gratis, Rating {rating})")

print("\n ANALISIS ")
if dp_rating > gr_rating:
    print("DP menghasilkan solusi yang lebih optimal dibanding Greedy.")
elif dp_rating == gr_rating:
    print("Kedua algoritma menghasilkan kualitas solusi yang sama.")
else:
    print("Greedy lebih optimal pada dataset ini.")

print(f"DP lebih lambat {dp_time/gr_time:.1f}x dari Greedy")

Masukkan budget maksimal (dalam ribuan): 100000

 HASIL PERBANDINGAN DP vs GREEDY 

Dynamic Programming:
Total Rating: 109.70, Total Cost: Rp 847000, Time: 1.64903s
Destinasi: Solo_Safari, Museum_Tumurun, De_Tjolomadoe, Benteng_Vastenburg, Pura_Mangkunegaran, Gedung_Djoeang_45, Taman_Balekambang, Lokananta_Bloc, Museum_Keris_Nusantara, Rasamadu_Heritage, Kemuning_Sky_Hills, Air_Terjun_Jumog, Air_Terjun_Grojogan_Sewu, Pracima_Tuin, House_Of_Danar_Hadi, Keraton_Surakarta, Candi_Cetho, Solo_Summerland_Waterpark, Wayang_Orang_Sriwedari, Kale_Locale, Museum_Radya_Pustaka, Tawangmangu_Park, Candi_Sukuh, The_Haritage_Palace

Greedy:
Total Rating: 109.70, Total Cost: Rp 847000, Time: 0.00011s
Destinasi: Museum_Keris_Nusantara, Taman_Balekambang, Museum_Tumurun, Lokananta_Bloc, Kale_Locale, Candi_Sukuh, Pura_Mangkunegaran, Candi_Cetho, House_Of_Danar_Hadi, Pracima_Tuin, Keraton_Surakarta, Wayang_Orang_Sriwedari, Kemuning_Sky_Hills, Benteng_Vastenburg, Gedung_Djoeang_45, Air_Terjun_Jumog, De_Tjo